In [106]:
import pandas as pd

In [107]:
df = pd.read_csv('./results/raw_shipment_classification_dataset.csv')

In [108]:
df.duplicated().sum()

np.int64(22)

In [109]:
df.drop_duplicates(inplace=True)

In [110]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 802 entries, 0 to 823
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   shipped_date            802 non-null    object 
 1   eta                     802 non-null    object 
 2   receipt_date            802 non-null    object 
 3   delay_days              802 non-null    int64  
 4   shipping_duration_days  802 non-null    int64  
 5   lead_time_days          802 non-null    int64  
 6   is_early_delivery       802 non-null    int64  
 7   coo                     802 non-null    object 
 8   scac                    802 non-null    object 
 9   tariff_amount           802 non-null    float64
 10  ocean_freight           802 non-null    float64
 11  delivery_terms          802 non-null    object 
 12  po_shipment_terms       802 non-null    object 
 13  tariff_type             802 non-null    object 
 14  total_bcy               802 non-null    object 

In [111]:
date_time_columns = ['shipped_date', 'eta', 'receipt_date']
for col in date_time_columns:
    df[f'{col}'] = pd.to_datetime(df[f'{col}'], errors='raise')

df['shipped_date_weekday'] = df['shipped_date'].dt.weekday
df['shipped_date_month'] = df['shipped_date'].dt.month
df['shipped_date_day'] = df['shipped_date'].dt.day

df.drop(columns=date_time_columns, inplace=True)

In [112]:
# 1. Remove 'USD' and any surrounding whitespace
df['total_bcy'] = df['total_bcy'].str.replace('USD', '').str.strip()

# 2. FIX: Remove all thousands separators (commas)
df['total_bcy'] = df['total_bcy'].str.replace(',', '')

# 3. Convert the clean string to float
df['total_bcy'] = df['total_bcy'].astype(float)

In [113]:
numeric_cols = [
    'delay_days',
    'shipping_duration_days', 'lead_time_days',
    'quantity_in', 'total_bcy', 'vendor_avg_delay_days', 
    'vendor_shipments', 'vendor_on_time_rate', 'vendor_p50_delay_days', 
    'vendor_p90_delay_days'
]


for col in numeric_cols:
    df = df[df[col] >= 0]
    

In [114]:
# Convert total_bcy to numeric
df['total_bcy'] = pd.to_numeric(df['total_bcy'], errors='coerce')


In [115]:
distance_dict = {
    'INDIA': 11000,
    'CHINA': 6000,
    'INDONESIA': 8200,
    'VIETNAM': 6200,
    'ECUADOR': 2100,
    'THAILAND': 8100
}
df['coo'].value_counts()

coo
INDIA        666
CHINA         50
INDONESIA     39
VIETNAM       39
ECUADOR        5
THAILAND       3
Name: count, dtype: int64

In [116]:
df['distance_nm'] = df['coo'].map(distance_dict)
df.drop(columns=['coo','item_product_category'], inplace=True)

In [117]:
df.columns

Index(['delay_days', 'shipping_duration_days', 'lead_time_days',
       'is_early_delivery', 'scac', 'tariff_amount', 'ocean_freight',
       'delivery_terms', 'po_shipment_terms', 'tariff_type', 'total_bcy',
       'quantity_in', 'item_sku', 'item_brand', 'item_manufacturer',
       'item_size', 'vendor_name', 'vendor_avg_delay_days', 'vendor_shipments',
       'vendor_on_time_rate', 'vendor_p50_delay_days', 'vendor_p90_delay_days',
       'shipped_date_weekday', 'shipped_date_month', 'shipped_date_day',
       'distance_nm'],
      dtype='object')

In [118]:
df.to_csv('./results/cleaned_shipment_classification_dataset.csv', index=False)

In [119]:
# df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()
label_encoder = LabelEncoder()

# Ensure categorical features are strings (clean)
categorical_cols = [
    'scac', 'delivery_terms',
    'po_shipment_terms', 'tariff_type', 'item_brand',
    'item_manufacturer', 
    'item_size', 'vendor_name'
]

for col in categorical_cols:
    df_encoded[col] = df_encoded[col].astype(str).str.strip().replace('', 'Unknown')

for col in categorical_cols:
    df_encoded[col] = label_encoder.fit_transform(df_encoded[col].astype(str))


In [120]:
from sklearn.preprocessing import StandardScaler

columns_to_scale = [
    "tariff_amount",
    "quantity_in",
    "ocean_freight",
]

scaler = StandardScaler()
df_encoded[columns_to_scale] = scaler.fit_transform(df_encoded[columns_to_scale])